# 02 — Bayesian Logistic Regression

Specify priors based on domain knowledge, build the generative model in PyMC, and run MCMC sampling to get the posterior.

## Prior elicitation

Before touching PyMC, we decide what we *believe* about each feature.

| Feature | Prior belief | Expected sign |
|---|---|---|
| tenure | Longer tenure → loyal customer | Negative |
| Contract: month-to-month | No lock-in → easy to leave | Positive |
| MonthlyCharges | Higher bill → more incentive to shop around | Positive |
| Contract: two-year | Locked in → unlikely to churn | Negative |
| Fiber optic | Competitive market, higher bills | Positive |
| SeniorCitizen | Slightly more vulnerable to churn | Positive |

All betas: `Normal(0, 1)` — weakly informative on a standardised scale.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import warnings; warnings.filterwarnings("ignore")

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()
df = pd.get_dummies(df, columns=["Contract", "InternetService"], drop_first=False)
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract_Month_to_month",
    "Contract_Two_year",
    "InternetService_Fiber_optic",
    "SeniorCitizen",
]

X = StandardScaler().fit_transform(df[features])
y = df["Churn"].values
print(f"X: {X.shape}, y: {y.shape}, churn rate: {y.mean():.3f}")

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


X: (7032, 7), y: (7032,), churn rate: 0.266


## Build the model

In [2]:
with pm.Model() as churn_model:
    # Priors — weakly informative
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    betas = pm.Normal("betas", mu=0, sigma=1, shape=X.shape[1])

    # Linear combination → probability via sigmoid
    logit_p = alpha + pt.dot(X, betas)
    p = pm.Deterministic("p", pm.math.sigmoid(logit_p))

    # Likelihood
    observed = pm.Bernoulli("churn", p=p, observed=y)

try:
    pm.model_to_graphviz(churn_model)
except Exception:
    print("graphviz not installed — skipping model graph. Run: pip install graphviz")
    print("Model structure: alpha (intercept) + betas (7 features) -> sigmoid -> Bernoulli")

## Sample the posterior

2000 draws × 2 chains = 4000 posterior samples. Expect ~4 min on CPU.

In [3]:
import pymc as pm
import pytensor
import jax

print("JAX devices:", jax.devices())  # should show your GPU here

with churn_model:
    trace = pm.sample(
        2000, tune=1000, chains=2,
        target_accept=0.9,
        return_inferencedata=True,
        progressbar=True,
        nuts_sampler="numpyro",   # ← uses JAX/CUDA under the hood
    )

az.to_netcdf(trace, "../outputs/trace.nc")
print("Trace saved.")

RuntimeError: jaxlib version 0.10.0 is newer than and incompatible with jax version 0.4.21. Please update your jax and/or jaxlib packages.

In [1]:
import jax, jaxlib
print(jax.__version__)    # 0.10.0
print(jaxlib.__version__) # 0.10.0
print(jax.devices())      # should show your NVIDIA GPU

RuntimeError: jaxlib version 0.10.0 is newer than and incompatible with jax version 0.4.21. Please update your jax and/or jaxlib packages.

## Posterior summary

In [ ]:
summary = az.summary(trace, var_names=["alpha", "betas"])
summary.index = ["alpha"] + features
summary[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat"]]